# **Libraries**

In [ ]:
%pip install "sagemaker<3"
import boto3, sagemaker, json
from sagemaker.sklearn.model import SKLearnModel
from sagemaker.model_monitor import DataCaptureConfig

Note: you may need to restart the kernel to use updated packages.
sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [ ]:
print(sagemaker.__version__)

2.257.7


# **Upload the Model to S3**

In [ ]:
session = sagemaker.Session()
bucket = session.default_bucket()
prefix = 'loss_cost_lgbm_v3'

model_s3_uri = session.upload_data(
    path = 'loss_cost_lgbm.tar.gz',
    bucket = bucket,
    key_prefix = prefix
)

#print(model_s3_uri)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [ ]:
%cat code/inference.py

import os, json
import pandas as pd
import lightgbm as lgb
from features import build_features

def model_fn(model_dir):

    return lgb.Booster(model_file = os.path.join(model_dir, 'loss_cost_lgbm.txt'))

def input_fn(request_body, content_type):

    if content_type != 'application/json':
        raise ValueError('Unsupported content type')
        
    raw = json.loads(request_body)
    features = build_features(pd.DataFrame([raw]))
    return {'features': features}

def predict_fn(inputs, model):

    preds = model.predict(inputs['features'])
    return {'loss_cost' : preds.tolist(),
           'engineered_features': inputs['features'].to_dict(orient = 'records')}

def output_fn(prediction, accept):

    return json.dumps(prediction), 'application/json'

In [ ]:
%cat code/requirements.txt

lightgbm==4.5.0
scikit-learn
scipy 
numpy
pandas 

# **Data Capture**

We will use SageMaker's built in **Data Capture Feature** to persist features and predictions for monitoring and retraining jobs.

In [ ]:
capture_s3_uri = f"s3://{bucket}/{prefix}/capture/"
print(capture_s3_uri)

data_capture_config = DataCaptureConfig(
    enable_capture=True,
    sampling_percentage=100,
    destination_s3_uri=capture_s3_uri,
    capture_options=["REQUEST", "RESPONSE"],
    csv_content_types=["text/csv"],
    json_content_types=["application/json"],
)

s3://amazon-sagemaker-049428442670-us-east-1-5yk3gphwq9r5t3/loss_cost_lgbm_v3/capture/
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


# **Deploy Model**

In [ ]:
role = sagemaker.get_execution_role()

model = SKLearnModel(
    model_data=model_s3_uri,
    role=role,
    entry_point="inference.py",
    source_dir="code",
    framework_version="1.2-1",
    py_version="py3",
)

predictor = model.deploy(
    initial_instance_count=1,
    instance_type="ml.m5.large",
    endpoint_name="lgbm-loss-cost-endpoint-v4",
    data_capture = data_capture_config
)

print("Endpoint:", predictor.endpoint_name)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
------!Endpoint: lgbm-loss-cost-endpoint-v4


In [ ]:
# check capture
sm = boto3.client("sagemaker")
ep = sm.describe_endpoint(EndpointName="lgbm-loss-cost-endpoint-v4")
print(ep["EndpointStatus"], ep.get("DataCaptureConfig"))

InService None


In [ ]:
predictor = sagemaker.predictor.Predictor(
    endpoint_name="lgbm-loss-cost-endpoint-v4",
    sagemaker_session=session,
)

predictor.update_data_capture_config(data_capture_config=data_capture_config)
print("capture update requested")

------!capture update requested


In [ ]:
# check capture
sm = boto3.client("sagemaker")
ep = sm.describe_endpoint(EndpointName="lgbm-loss-cost-endpoint-v4")
print(ep["EndpointStatus"], ep.get("DataCaptureConfig"))

InService {'EnableCapture': True, 'CaptureStatus': 'Started', 'CurrentSamplingPercentage': 100, 'DestinationS3Uri': 's3://amazon-sagemaker-049428442670-us-east-1-5yk3gphwq9r5t3/loss_cost_lgbm_v3/capture/'}


# **Invoke the Endpoint**

In [ ]:
payload = {
    "RiskArea": 5.0,
    "BonusMalus": 100,
    "DrivAge": 35,
    "LicAge": 120,
    "VehUsage": "Private"
}

runtime = boto3.client('sagemaker-runtime', region_name = 'us-east-1')

resp = runtime.invoke_endpoint(
    EndpointName = 'lgbm-loss-cost-endpoint-v4',
    ContentType = 'application/json',
    Body = json.dumps(payload)
)

print(json.loads(resp['Body'].read()))

{'loss_cost': [570.4891651846364], 'engineered_features': [{'RiskArea': 5.0, 'bm_centered': 0, 'bm_bonus': 0, 'bm_penality': 0, 'bm_is_bonus': 0, 'bm_is_penality': 0, 'driv_age_sq': 1225, 'inv_driv_age': 0.02857142857142857, 'young': 0, 'senior': 0, 'licage_years': 10.0, 'first_year': 0, 'first_two_years': 0.0, 'licage_log': 2.3978952727983707, 'experience_gap': 7.0, 'late_starter': 1, 'young_x_risk_area': 0.0, 'young_x_licAge_fst_2': 0.0, 'bm_penality_x_risk_area': 0.0, 'bm_x_age': 0, 'bm_to_licage': 0.0, 'bm_to_age': 0.0, 'VehUsage_lgb': 'Private'}]}


# **Read Capture**

In [ ]:
s3 = boto3.client("s3")
cap_prefix = "loss_cost_lgbm_v3/capture/"

resp = s3.list_objects_v2(Bucket=bucket, Prefix=cap_prefix)
print(resp.get("KeyCount"))
for o in resp.get("Contents", []):
    print(o["Key"], o["Size"])

1
loss_cost_lgbm_v3/capture/lgbm-loss-cost-endpoint-v4/AllTraffic/2026/11/03/21/05-55-054-f672106e-5cc0-4b25-b349-f487c10e50ef.jsonl 2120
